# 🚀 Ultimate AI Audit Server (YOLOv8x + BLIP-Large + ViT AI-Detector)

This server provides professional-grade AI analysis for your Social Media App.

### Models Used:
1. **Object Detection**: YOLOv8x (Extra Large) - Highest accuracy available.
2. **Captioning**: BLIP-Large - Superior descriptive capability.
3. **Deepfake/AI Detection**: ViT AI-Image-Detector - Real-time analysis of image authenticity with exact percentages.

### Instructions:
1. Ensure **Runtime Type** is set to **T4 GPU**.
2. Enter your **Ngrok Authtoken** and **Static Domain** in Step 4.
3. Run all cells and copy the URL to your Flutter app.

### Step 1: Install Professional AI Dependencies

In [ ]:
!pip install Flask flask-cors ultralytics transformers torch pillow timm pyngrok

### Step 2: Write Advanced AI Server (`app.py`)

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image, ImageOps
import io, base64, torch, os, logging
import numpy as np
import cv2
from ultralytics import YOLO
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline

app = Flask(__name__)
CORS(app)

API_KEY = "social_media_app_2024_secure_key"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n--- INITIALIZING ADVANCED MODELS ON {device.upper()} ---")

# 1. YOLOv8x for maximum object detection accuracy
obj_model = YOLO('yolov8x.pt')

# 2. BLIP-Large for high-quality captions
cap_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
cap_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)

# 3. ViT AI-Detector for Deepfake/AI detection (High Accuracy)
fake_detector = pipeline("image-classification", model="umm-maybe/AI-image-detector", device=0 if device=="cuda" else -1)

print("✅ All Professional Models Loaded Successfully!")

def process_image(file):
    img = Image.open(file).convert('RGB')
    return ImageOps.exif_transpose(img)

@app.route('/process_all', methods=['POST'])
def process_all():
    if request.headers.get('X-API-Key') != API_KEY: return jsonify({'error': 'Unauthorized'}), 401
    if 'image' not in request.files: return jsonify({'error': 'No image'}), 400
    
    img = process_image(request.files['image'])
    img_array = np.array(img)
    
    # --- 1. Object Detection (YOLOv8x) ---
    results = obj_model(img_array, conf=0.25, verbose=False)
    detections = [{'label': obj_model.names[int(box.cls[0])].capitalize()} for box in results[0].boxes]
    marked_img_bgr = results[0].plot()
    marked_img_rgb = cv2.cvtColor(marked_img_bgr, cv2.COLOR_BGR2RGB)
    
    buf = io.BytesIO()
    Image.fromarray(marked_img_rgb).save(buf, format="JPEG")
    encoded_img = base64.b64encode(buf.getvalue()).decode()
    
    # --- 2. Image Captioning (BLIP-Large) ---
    inputs = cap_processor(img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = cap_model.generate(**inputs, max_new_tokens=60)
    caption = cap_processor.decode(out[0], skip_special_tokens=True).capitalize()
    
    # --- 3. Deepfake/AI Detection (ViT) ---
    fake_results = fake_detector(img)
    ai_score = 0
    real_score = 0
    for r in fake_results:
        if r['label'].lower() == 'ai': ai_score = r['score'] * 100
        if r['label'].lower() == 'human': real_score = r['score'] * 100
    
    verdict = "Human Captured" if real_score > ai_score else "AI Generated"
    if ai_score > 85: verdict = "Synthetic Deepfake"
    
    return jsonify({
        'objects': detections,
        'caption': caption,
        'marked_image': encoded_img,
        'deepfake': {
            'is_real': real_score > ai_score,
            'real_score': round(real_score, 2),
            'ai_score': round(ai_score, 2),
            'verdict': verdict
        }
    })

# Individual routes for compatibility
@app.route('/detect_objects', methods=['POST'])
def d_obj(): return process_all()

@app.route('/generate_caption', methods=['POST'])
def g_cap(): return process_all()

@app.route('/detect_deepfake', methods=['POST'])
def d_fake(): return process_all()

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)


### Step 3: Run Server

In [ ]:
import subprocess, time
with open("flask.log", "w") as f:
    subprocess.Popen(["python", "app.py"], stdout=f, stderr=f)
print("⏳ Loading XL Models (30-60 seconds)...")
time.sleep(45)
print("✅ Server is starting.")

### Step 4: Persistent Ngrok Tunnel

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = "3E6uCYTOIwLKpu6Og19WcYgeJHR_2EzCmDF29YpsZ4A4PbDMg"
NGROK_STATIC_DOMAIN = "eatable-monument-gone.ngrok-free.dev"

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(5000, domain=NGROK_STATIC_DOMAIN)
print(f"\n🚀 API LIVE: {tunnel.public_url}")